# <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#005B46; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #003300"> Introduction</p>

This notebook explains what the competition actually runs, looks at the data and the starter kit, fixes three problems in the starter kit, and builds my own agent into a `submission.zip`.

Worth knowing up front:

- The unmodified starter kit (`sample_submission/`) **errored** when I submitted it: *"Your notebook hit an unhandled error while rerunning your code"*.
- Its `eval_config.yaml` caps every task at **1 minute and 10 tool calls**.
- Its two LoRA adapters are **no-ops**: every `lora_B` weight is zero.
- With those fixed and 5 minutes per task, it runs but scores **0.00**, so this version builds my own agent (see *Version 2: my own agent*).
- The platform allows **1 submission per day** (the rules page says 5).
- Half of the graph and embedding files are **empty**, even in the copy mounted on Kaggle.
- The code graphs contain **no async functions**, and embedding similarity is a weak signal.

## <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#005B46; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #003300">What you submit and how it is scored</p>

- You submit a zip holding a **declarative Google ADK agent**: YAML configs, Markdown prompts, and optionally LoRA adapters and skills. There is no Python entry point.
- Kaggle's harness compiles it, serves the Gemma 4 model you declare with vLLM on **4x NVIDIA L4** (32k context), and runs the agent on about **120 hidden tasks from private repositories**.
- Each task runs in two phases:
  1. **Agent phase.** The agent works in `/workspace` (the repo at the task's base commit, offline) with 9 tools: `run_command`, `read_file`, `edit_file`, `write_file`, `get_status`, `submit_patch`, `search_similar_code`, `get_code_neighbors`, `get_code_subgraph`. Its patch is the `git diff HEAD` of the working tree.
  2. **Verification phase.** A fresh container applies the patch and the task's hidden tests. The task passes if pytest exits with code 0.
- The score is the share of tasks that pass.
- Limits: **12 hours for all tasks**, including sandbox setup; per-task limits in `eval_config.yaml`; under 3 GiB unpacked; one base model per submission.

The harness libraries (`swegemma`, `adk-submission`, `adk-eval-core`) are documented in `HARNESS_README.md` but not published, so there is no official way to evaluate locally.

In [ ]:
import json
import os
import re
import shutil
import struct
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml


def find_competition_dir() -> Path:
    if override := os.environ.get("G4_COMP_DIR"):
        return Path(override)
    matches = sorted(Path("/kaggle/input").glob("**/tasks.jsonl"))
    if not matches:
        raise FileNotFoundError("Attach the data: Add Input > Competitions > gemma-4-developer-agent")
    return matches[0].parent


COMP = find_competition_dir()
WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("G4_WORKING_DIR", "working"))
WORKING.mkdir(parents=True, exist_ok=True)
print(COMP)
print(sorted(p.name for p in COMP.iterdir()))

## <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#005B46; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #003300">The data at a glance</p>

In [ ]:
tasks = pd.read_json(COMP / "tasks.jsonl", lines=True)
print(f"{len(tasks)} tasks, {tasks.base_commit.nunique()} distinct base commits, "
      f"{tasks.hints_text.str.strip().ne('').sum()} with hints")
tasks.repo.value_counts().rename("tasks").to_frame()

In [ ]:
def patched_files(patch: str) -> list[str]:
    return re.findall(r"^\+\+\+ b/(\S+)", patch, re.MULTILINE)


pd.DataFrame({
    "problem statement chars": tasks.problem_statement.str.len(),
    "files in reference fix": tasks.patch.map(lambda p: len(patched_files(p))),
    "files in test patch": tasks.test_patch.map(lambda p: len(patched_files(p))),
}).describe().loc[["mean", "50%", "max"]].round(1)

In [ ]:
example = tasks.set_index("instance_id").loc["rich_3006"]
print(example.problem_statement[:1500])
print("\nReference fix touches:", patched_files(example.patch))
print("Test patch touches:", patched_files(example.test_patch))

Things to notice:

- Problem statements can be pull-request descriptions rather than issue reports, checklist and all, like the one above.
- 129 training tasks from 4 public repos. The hidden test set comes from private repos, so the score measures how well the agent generalizes, not how well it knows fastapi.
- `hints_text` is empty for every task.
- `patch` (the reference fix) and `test_patch` (the verification tests) are there for training and local checks. Every patch in `tasks.jsonl` lacks its final newline; add one before applying it with `git apply` or `patch`.
- `snapshots/<instance_id>.tgz` is the repo at the base commit. `graphs/` and `embeddings/` back the three code-intelligence tools (see the graph-tools section).

## <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#005B46; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #003300"> The starter kit, file by file </p>

In [ ]:
SAMPLE = COMP / "sample_submission"
for path in sorted(SAMPLE.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size:>8,}  {path.relative_to(SAMPLE)}")

In [ ]:
for name in ["agent.yaml", "sub_agents/code_analyzer.yaml", "configs/sampling.yaml", "eval_config.yaml"]:
    print(f"--- {name}\n{(SAMPLE / name).read_text()}")

- `agent.yaml` is the root agent: the model alias (`gemma-4-31b-it-qat-w4a16-ct`, Gemma 4 31B with 4-bit weights), a LoRA adapter, the system prompt (`!include prompts/system.md`), all 9 tools, and a sub-agent exposed as a tool.
- `sub_agents/code_analyzer.yaml` is a read-only analyzer. Wrapping it in `agent_tool` with `skip_summarization: true` keeps its file reads out of the root agent's context window.
- `configs/sampling.yaml` sets temperature 0.2 and thinking with a 4,096-token budget.
- `eval_config.yaml` sets the per-task budget.

## <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#005B46; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #003300">Three fixes before submission </p>

### The LoRA adapters do nothing

A LoRA adapter adds `B @ A` to a weight matrix, so if every value in `B` is zero, the adapter changes nothing. A safetensors file is an 8-byte header length, a JSON header, then raw tensor bytes, so we can check without extra libraries.

In [ ]:
def lora_tensors(path: Path) -> dict[str, np.ndarray]:
    raw = path.read_bytes()
    header_len = struct.unpack("<Q", raw[:8])[0]
    header = json.loads(raw[8 : 8 + header_len])
    body = raw[8 + header_len :]
    tensors = {}
    for name, info in header.items():
        if name == "__metadata__":
            continue
        assert info["dtype"] == "BF16", info["dtype"]
        start, end = info["data_offsets"]
        bf16_bits = np.frombuffer(body[start:end], dtype=np.uint16).astype(np.uint32) << 16
        tensors[name] = bf16_bits.view(np.float32).reshape(info["shape"])
    return tensors


for adapter in sorted((SAMPLE / "adapters").iterdir()):
    for name, tensor in lora_tensors(adapter / "adapter_model.safetensors").items():
        print(f"{adapter.name:10s} {name.split('layers.')[1]:34s} shape={str(tensor.shape):12s} all zero: {not tensor.any()}")

Every `lora_B` is zero, so both adapters are placeholders (rank 4, layer 0 only, byte-identical to each other). Dropping them and their `adapter:` lines leaves the model's behaviour unchanged and removes one thing that can fail at scoring time.

### The 1-minute budget

The sample `eval_config.yaml` gives each task 1 minute and 10 tool calls. The scorer allows 12 hours for all tasks, including sandbox setup.

In [ ]:
hidden_tasks, hours = 120, 12
print(f"{hours * 60 / hidden_tasks:.0f} minutes per task if tasks run one at a time")

How many tasks the scorer runs at once is not documented. My first version gave each task 5 minutes (with `timeout_seconds: 300` and `max_tool_calls: 50`) and scored **0.00**, and the whole run finished in about 80 minutes, far inside the 12 hours. The two public notebooks that scored above 0 ship no `eval_config.yaml` at all, so version 2 drops the file and runs on the harness defaults (the README lists 60 minutes per task). The sample's own comment says the file controls "per-task execution budgets and sandbox command timeouts", so `timeout_seconds` is most likely the per-command timeout.

### Package files only, with `agent.yaml` at the root

The harness wants exactly one root config (`agent.yaml`, `agent.yml`, `root_agent.yaml` or `root_agent.yml`) at the root of the archive, only `.yaml`, `.yml`, `.md`, `.txt`, `.py`, `.json` and `.safetensors` files, and under 3 GiB unpacked. The zip built at the end of this notebook holds files only, with no directory entries.

Why change all three at once: my unmodified starter-kit submission errored, and the public notebooks that ran without errors at the time shipped no adapters, no `eval_config.yaml`, and files-only zips. Changing all three worked (the next run scored 0.00 without errors), but it does not say which change mattered.

## <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#005B46; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #003300">Graph tools: what they can and cannot do </p>

`get_code_neighbors(node)` and `get_code_subgraph(nodes)` query `graphs/<task>.json`; `search_similar_code(query)` ranks nodes by cosine similarity using `embeddings/<task>.npz`. The harness only offers these tools when both files are over 100 bytes.

Each task-named file is meant to be a hard link to a commit-named file (`<repo>_<base_commit>`), but one file of each pair is empty, both in the zip download and in the copy mounted in Kaggle notebooks. The helper below finds the readable copy.

In [ ]:
def readable(kind: str, suffix: str, task: pd.Series) -> Path:
    short = task.instance_id.rsplit("_", 1)[0]
    names = [task.instance_id, f"{short}_{task.base_commit}", *tasks.instance_id[tasks.base_commit == task.base_commit]]
    return next(p for p in (COMP / kind / f"{n}{suffix}" for n in names) if p.stat().st_size > 100)


empty = {kind: sum(p.stat().st_size == 0 for p in (COMP / kind).iterdir()) for kind in ["graphs", "embeddings"]}
print("empty files in this copy of the data:", empty)

In [ ]:
OWN_DEFINITION = re.compile(r"^\s*(async\s+def|def|class)\s", re.MULTILINE)


def is_async(text: str) -> bool:
    match = OWN_DEFINITION.search(text or "")
    return bool(match) and match.group(1).startswith("async")


task = tasks.set_index("instance_id", drop=False).loc["fastapi_13786"]
graph = json.loads(readable("graphs", ".json", task).read_text())
nodes = {node["id"]: node for node in graph["nodes"]}
print(f"{len(nodes):,} nodes, {len(graph['edges']):,} edges")
print("edge types:", pd.Series([edge["type"] for edge in graph["edges"]]).value_counts().to_dict())
print("async function nodes:", sum(is_async(node["text"]) for node in nodes.values()))
for symbol in ["fastapi.routing.get_request_handler", "fastapi.routing.serialize_response"]:
    print(f"{symbol} in graph: {symbol in nodes}")

In [ ]:
embeddings = np.load(readable("embeddings", ".npz", task))
keys = embeddings.files
vectors = np.stack([embeddings[key] for key in keys]).astype(np.float64)
unit = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


def similar(symbol: str, k: int = 5) -> pd.DataFrame:
    query = next(key for key in keys if key == symbol or key.endswith("." + symbol))
    scores = unit @ unit[keys.index(query)]
    top = np.argsort(-scores)[1 : k + 1]
    print("query:", query)
    return pd.DataFrame({"node": [keys[i] for i in top], "cosine": scores[top].round(3)})


rng = np.random.default_rng(0)
i, j = rng.integers(0, len(keys), (2, 20_000))
print(f"median cosine between random pairs of nodes: {np.median((unit[i] * unit[j]).sum(axis=1)):.2f}")
similar("APIKeyHeader")

- The only edge type is `calls`: no import or containment edges, and no module nodes.
- Async functions and methods are missing entirely (first reported on the forum by Jason Wold https://www.kaggle.com/competitions/gemma-4-developer-agent/discussion/742911). Much of fastapi's request path is async, so the graph tools cannot reach it, while `grep` and `read_file` can.
- `search_similar_code` takes a symbol name, not a sentence. Offline, the query is matched to an existing node, then ranked by cosine similarity.
- The embeddings are dominated by one direction, so even random pairs of nodes score high, and top matches can be unrelated code. Treat the tool as a weak hint.

## <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#005B46; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #003300">Version 2: my own agent</p>

Results so far on the public leaderboard (about 60 tasks):

| Submission | Public score |
| :--- | :--- |
| Starter kit, unmodified | error |
| Starter kit with the three fixes, 5 minutes per task | 0.00 |
| [Black Cat SWE Agent](https://www.kaggle.com/code/lucifer19/black-cat-swe-agent-second-strike) by lucifer19 | 0.06 |
| [GEMMA: EDA, Baseline for a start](https://www.kaggle.com/code/romanrozen/gemma-eda-baseline-for-a-start-lb-top-1) by romanrozen | 0.12 |

Version 2 changes several things at once, based on what the two scoring notebooks have in common:

| Change | Why |
| :--- | :--- |
| No `eval_config.yaml`, so default budgets | Both scoring notebooks ship none; my 5-minute run scored 0.00 |
| Thoughts not returned (`include_thoughts: false`), output capped at 8,192 tokens | Shared by both scoring notebooks |
| A coder plus a read-only `code_analyzer` sub-agent that answers in a fixed format | The design of the 0.12 notebook, and the answer format follows romanrozen's; unlike that notebook, the coder keeps all nine tools, as in the starter kit |
| New prompts | The usual loop (reproduce, `py_compile`, targeted tests, `git diff` review), plus facts from this notebook: use `git grep` because the sandbox has no `rg` or `tree`, the working tree is graded even without `submit_patch()`, and the graph tools cannot see async code. The rules for pre-existing test failures are adapted from the starter kit's prompt |

Because several things change at once, a score will not say which change mattered.

In [ ]:
V2_FILES = {
    "agent.yaml": r"""name: swe_coder
model: gemma-4-31b-it-qat-w4a16-ct
description: Fixes one repository issue with a small, verified source patch.
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
  - get_code_neighbors
  - search_similar_code
  - get_code_subgraph
  - agent_tool:
      config_path: sub_agents/code_analyzer.yaml
      skip_summarization: true
generate_content_config: !include configs/sampling.yaml
""",
    "sub_agents/code_analyzer.yaml": r"""name: code_analyzer
model: gemma-4-31b-it-qat-w4a16-ct
description: Read-only code navigator. Given an issue, returns where to fix it, the root cause and a fix plan.
instruction: !include ../prompts/analyzer.md
tools:
  - run_command
  - read_file
  - get_code_neighbors
  - get_code_subgraph
  - search_similar_code
generate_content_config: !include ../configs/sampling.yaml
""",
    "configs/sampling.yaml": r"""temperature: 0.2
top_p: 0.95
top_k: 40
max_output_tokens: 8192
seed: 42
thinking_config:
  thinking_budget: 4096
  include_thoughts: false
""",
    "prompts/system.md": r"""You are an autonomous software engineer fixing one issue in the Python repository at /workspace. No one will answer questions. Keep working with your tools until the fix is in place, then call `submit_patch()`.

## How you are graded
- Hidden tests run on your changes in a fresh checkout. Only source changes in the working tree count, and a careful best-effort fix beats no patch.
- If time runs out, the working tree is graded as it is, so never revert a plausible fix.
- Test files used by the hidden tests are reset before grading, so editing tests never helps. Never edit tests, `pytest.ini` or `conftest.py`.

## Environment
- Offline, with dependencies installed. Do not install packages.
- /workspace is a git repository at the base commit.
- Available: git, grep, find, sed, awk, python3. Not available: rg (ripgrep), tree.
- Commands time out after 300 seconds and output is cut at 5,000 characters. `read_file` returns at most 150 lines, so read focused ranges.
- `get_status()` and `submit_patch()` are free.

## Workflow
1. Understand. Work out the expected and the actual behaviour. Some issues are pull-request descriptions: skip checklists and template text, keep the substance. Note exact names, messages and values.
2. Localize. Call `code_analyzer` with the issue text; it replies with LOCATION, ROOT CAUSE and FIX PLAN. Confirm by reading those lines. If it is unsure or wrong, search yourself: `git grep -n "<identifier>" -- '*.py' | head -30`.
3. Reproduce. Write a minimal script to /tmp/repro.py that shows the problem, and run it.
4. Fix. Edit with `edit_file`: copy `old_string` exactly from the file, including indentation, and keep it short but unique. Fix the root cause, handle the edge cases the issue names, and keep public signatures unchanged. Documentation examples under `docs_src/` count as source when the issue concerns them.
5. Verify. After each edit run `python3 -m py_compile <file>` and rerun /tmp/repro.py, then run the targeted tests described below.
6. Submit. Check `git status --short` and `git diff`, remove anything unintended, then call `submit_patch()` as your last tool call and reply with one line on what changed.

## Tests
- Run only the test file or test that exercises your change: `python3 -m pytest tests/test_x.py -q -x -k <name>`, or `python3 -m unittest tests.test_x`. Find test files with `git ls-files | grep test_<name>`.
- Never run bare `pytest`, `pytest .` or `python3 -m unittest discover`. A whole suite can take minutes and use up your time.
- Existing tests may already fail for reasons unrelated to the issue, such as missing fixtures, no network access or import errors. Ignore those failures: do not fix them, stub them or change test code. Judge your fix by the tests that exercise it.

## Code-graph tools
- `get_code_neighbors` and `get_code_subgraph` show synchronous call relationships only: the graph has only "calls" edges and no async functions.
- `search_similar_code` takes a symbol name such as `APIKeyHeader`, not a sentence, and its ranking is imprecise. Confirm every hit by reading the code.
- If a graph tool returns an error, stop using it. For async code and everything else, use `git grep` and `read_file`.

## Pace
- Call `get_status()` every 8 or so tool calls. When a quarter of the time remains, stop exploring and go straight to fix, verify and submit.
- Keep outputs short: `head`, `grep -n`, `pytest -q`.
- If an edit fails twice, reread the exact lines and retry with a smaller snippet. Keep each edit small so the tool call is not cut off.
""",
    "prompts/analyzer.md": r"""You are `code_analyzer`, a read-only code navigator. You never modify files. Given an issue, find exactly where it must be fixed.

## Tools
- `run_command` for read-only commands only: `git grep -n`, `grep -rn`, `sed -n 'START,ENDp' FILE`, `ls`, `git log -p -S "<text>"`. ripgrep (rg) is not installed.
- `read_file` with tight line ranges to confirm what you found.
- `get_code_neighbors` and `get_code_subgraph` for synchronous call relationships. The graph has only "calls" edges and no async functions.
- `search_similar_code` takes a symbol name such as `APIKeyHeader` or `routing.get_request_handler`, not a sentence, and its ranking is imprecise. Confirm every hit by reading the code.
- If a graph tool returns an error, stop using it and search the source instead.

## Method
1. Pull the identifiers out of the issue: function and class names, error messages, file paths, options. Pull-request descriptions may include template text; ignore it.
2. Search for each one, then follow the code until you reach the line where the behaviour diverges from what the issue expects.
3. Confirm by reading the code. Never guess line numbers.

## Answer (at most 250 words, nothing else)
LOCATION: <path>:<start>-<end> (<function or class>)
ROOT CAUSE: <one or two sentences>
FIX PLAN: <the concrete change>
RELATED: <other places needing the same change, or "none">
TESTS: <existing test files that exercise this code>
CONFIDENCE: high | medium | low
""",
}

for name in ["prompts/system.md", "prompts/analyzer.md"]:
    print(f"--- {name}\n{V2_FILES[name]}")

## <p style="font-family:JetBrains Mono; font-weight:bold; letter-spacing: 2px; color:#005B46; font-size:140%; text-align:left;padding: 0px; border-bottom: 3px solid #003300"> Build, validate, and submit </p>

In [ ]:
BUNDLE = WORKING / "submission_bundle"
shutil.rmtree(BUNDLE, ignore_errors=True)
for name, text in V2_FILES.items():
    path = BUNDLE / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text)
for name in ["agent.yaml", "sub_agents/code_analyzer.yaml", "configs/sampling.yaml"]:
    print(f"--- {name}\n{(BUNDLE / name).read_text()}")

In [ ]:
ROOT_CONFIGS = ["agent.yaml", "agent.yml", "root_agent.yaml", "root_agent.yml"]
ALLOWED_SUFFIXES = {".yaml", ".yml", ".md", ".txt", ".py", ".json", ".safetensors"}
yaml.SafeLoader.add_constructor("!include", lambda loader, node: loader.construct_scalar(node))


def validate(bundle: Path) -> None:
    files = [p for p in bundle.rglob("*") if p.is_file()]
    roots = [name for name in ROOT_CONFIGS if (bundle / name).is_file()]
    assert len(roots) == 1, f"need exactly one root config, found {roots}"
    assert not any(p.is_symlink() for p in bundle.rglob("*")), "symlinks are rejected"
    assert all(p.suffix in ALLOWED_SUFFIXES for p in files), [p.name for p in files if p.suffix not in ALLOWED_SUFFIXES]
    assert sum(p.stat().st_size for p in files) < 3 * 2**30, "over 3 GiB unpacked"
    models = set()
    for config in (p for p in files if p.suffix in (".yaml", ".yml")):
        text = config.read_text()
        data = yaml.safe_load(text) or {}
        for target in re.findall(r"!include\s+(\S+)", text):
            assert (config.parent / target).is_file(), f"{config.name}: missing include {target}"
        if "adapter" in data:
            assert (bundle / "adapters" / data["adapter"] / "adapter_model.safetensors").is_file(), f"{config.name}: adapter {data['adapter']} not shipped"
        if "model" in data:
            models.add(data["model"])
    assert len(models) == 1, f"one base model per submission, found {models}"
    print(f"OK: {roots[0]} at the root, {len(files)} files, model {models.pop()}")


validate(BUNDLE)

In [ ]:
ZIP_PATH = WORKING / "submission.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(p for p in BUNDLE.rglob("*") if p.is_file()):
        archive.write(path, path.relative_to(BUNDLE).as_posix())

with zipfile.ZipFile(ZIP_PATH) as archive:
    names = archive.namelist()
assert "agent.yaml" in names and not any(name.endswith("/") for name in names)
shutil.rmtree(BUNDLE)
print(f"{ZIP_PATH} ({ZIP_PATH.stat().st_size:,} bytes)", *names, sep="\n  ")

**To submit:** if you want to try it yourself, click *Save Version* (Save & Run All). When the run finishes, open the version's *Output* tab and click *Submit*; the score then shows on this notebook. The platform allows one submission per day, resetting at 00:00 UTC.


### What to try next

- Model choice: `gemma-4-26b-a4b-it` activates about 4B parameters per token, so it may fit more turns into each task than the 31B.
- One agent or coder plus analyzer: the two scoring notebooks differ here, and 0.06 versus 0.12 is only about 4 versus 7 tasks, close to noise.
- When to use the graph tools versus `git grep` and `read_file`, given the gaps in the graph-tools section.

If you have any ideas, please share in the comments.